In [62]:
"""
We see the 

Always the csv file has 1 column with each row being only 1 value(ie no values separated by commas)
    ex: We have as 
        "Time:=0920,Tea"
    and we don't have as 
        "Time:=0920,Tea,Cinnamon"

So we find at a certain time, what were the items that users added to the cart.

And therefore, we show those items to users.

As an improvement, we can do this with conjunction of 'Using 'orders' to recommend items.ipynb'.
    Ie, we get some recommendations towards that user noh.
        We can show a certain item of that recommendation list, 
        in a certain time based on this 'Using 'cart_timestamp' to display items in certain times so that people add to cart .ipynb'
"""

'\nWe see the \n\nAlways the csv file has 1 column with each row being only 1 value(ie no values separated by commas)\n    ex: We have as \n        "Time:=0920,Tea"\n    and we don\'t have as \n        "Time:=0920,Tea,Cinnamon"\n\nSo we find at a certain time, what were the items that users added to the cart.\n\nAnd therefore, we show those items to users.\n\nAs an improvement, we can do this with conjunction of \'Using \'orders\' to recommend items.ipynb\'.\n    Ie, we get some recommendations towards that user noh.\n        We can show a certain item of that recommendation list, \n        in a certain time based on this \'Using \'cart_timestamp\' to display items in certain times so that people add to cart .ipynb\'\n'

# Stats of which items were bought in "Time"

In [63]:
from mlxtend.frequent_patterns import apriori,association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

def finding_association_rules(csv_file):
    data = {
        "Order":[

        ]
    }
    df_temp = pd.read_csv(csv_file)


    
    for i in range(0,len(df_temp)):
        value_ = df_temp.Order[i].split(',')
        data['Order'].append(value_)
    data
    
    df = pd.DataFrame(data)

    te = TransactionEncoder()
    te_ary = te.fit_transform(df['Order'])

    df_trans = pd.DataFrame(te_ary, columns = te.columns_)

    frequent_itemsets = apriori(df_trans, min_support=0.2, use_colnames=True)

    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.5)
    
    return frequent_itemsets, rules

csv_file = 'cart_timestamp.csv'
frequent_itemsets, rules = finding_association_rules(csv_file)

In [64]:
rules = rules[['antecedents','consequents','support','confidence']]
rules = rules.sort_values('confidence', ascending=False).drop_duplicates()
rules

,antecedents,consequents,support,confidence
0,(Biscuit),(Time:=0920),0.3,1.000000
2,(Chocolate),(Time:=0920),0.2,1.000000
4,(cinnamon),(Time:=0930),0.4,1.000000
5,(Time:=0930),(cinnamon),0.4,1.000000
1,(Time:=0920),(Biscuit),0.3,0.500000
3,(Time:=0920),(Chocolate),0.2,0.333333


In [65]:
"""
Order
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0930,Rice"
"Time:=0930,Rice"

"""

'\nOrder\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n\n'

In [66]:
time_only_ones = []
for i in rules['antecedents']:
    if(len(list(i)) == 1):
        if 'Time:=' in list(i)[0]:
            time_only_ones.append(list(i)[0])
time_only_ones = list(dict.fromkeys(time_only_ones))

In [67]:
time_only_dataframe = {
        'antecedents': [],
        'consequents': [],
        'support': [] ,
        'confidence': []
        }

time_only_dataframe = pd.DataFrame(time_only_dataframe)


for i in time_only_ones:
    rules_ = rules[rules['antecedents']==frozenset({i})]
    time_only_dataframe = pd.concat([time_only_dataframe, rules_])

In [68]:
sorted_by_support = time_only_dataframe.sort_values('support', ascending=False).drop_duplicates()
sorted_by_support

,antecedents,consequents,support,confidence
5,(Time:=0930),(cinnamon),0.4,1.000000
1,(Time:=0920),(Biscuit),0.3,0.500000
3,(Time:=0920),(Chocolate),0.2,0.333333


In [69]:
sorted_by_support_confidence = sorted_by_support.sort_values('confidence', ascending=False).drop_duplicates()
sorted_by_support_confidence

,antecedents,consequents,support,confidence
5,(Time:=0930),(cinnamon),0.4,1.000000
1,(Time:=0920),(Biscuit),0.3,0.500000
3,(Time:=0920),(Chocolate),0.2,0.333333


In [70]:
# See the code block below "# Test out with these difference scenarios" for sure.
#    It implements the "As an improvement,......" thing

# Test out with these difference scenarios

In [71]:
"""

Order
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Persian tea"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"

Order
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0920,Tea"


Order
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea"
"Time:=0920,Tea"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"



eventhough 0930->Rice is on top, the results show 0920 with it's and the reason is, we sort by support noh
Order
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0930,Rice"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea,Cinnamon"
"Time:=0920,Tea"
"Time:=0920,Tea"


"""

'\n\nOrder\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Persian tea"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n\nOrder\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n\n\nOrder\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n\n\n\neventhough 0930->Rice is on top, the results show 0920 with it\'s and the reason is, we sort by support noh\nOrder\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0930,Rice"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea,Cinnamon"\n"Time:=0920,Tea"\n"Time:=0920,Tea"\n\n\n'

# Let's implement the <br>

## "As an improvement,......" thing

In [72]:
# This is the list we get from 'Using 'orders' to recommend items.ipynb' noh

In [73]:
"""
Order
"Time:=0920,cinnamon"
"Time:=0920,cinnamon"
"Time:=0920,cinnamon"
"Time:=0920,cinnamon"
"Time:=0920,cinnamon"
"Time:=0920,cinnamon"
"Time:=0920,cinnamon"
"Time:=0920,Persian tea"
"Time:=0930,Biscuit"
"Time:=0930,Biscuit"
"Time:=0930,Biscuit"
"Time:=0930,Biscuit"



Order
"Time:=0920,Biscuit"
"Time:=0920,Biscuit"
"Time:=0920,Biscuit"
"Time:=0920,Biscuit"
"Time:=0920,Biscuit"
"Time:=0920,Biscuit"
"Time:=0920,Biscuit"
"Time:=0920,Biscuit"
"Time:=0920,Persian tea"
"Time:=0930,cinnamon"
"Time:=0930,cinnamon"
"Time:=0930,cinnamon"
"Time:=0930,cinnamon"




"""

'\nOrder\n"Time:=0920,cinnamon"\n"Time:=0920,cinnamon"\n"Time:=0920,cinnamon"\n"Time:=0920,cinnamon"\n"Time:=0920,cinnamon"\n"Time:=0920,cinnamon"\n"Time:=0920,cinnamon"\n"Time:=0920,Persian tea"\n"Time:=0930,Biscuit"\n"Time:=0930,Biscuit"\n"Time:=0930,Biscuit"\n"Time:=0930,Biscuit"\n\n\n\nOrder\n"Time:=0920,Biscuit"\n"Time:=0920,Biscuit"\n"Time:=0920,Biscuit"\n"Time:=0920,Biscuit"\n"Time:=0920,Biscuit"\n"Time:=0920,Biscuit"\n"Time:=0920,Biscuit"\n"Time:=0920,Biscuit"\n"Time:=0920,Persian tea"\n"Time:=0930,cinnamon"\n"Time:=0930,cinnamon"\n"Time:=0930,cinnamon"\n"Time:=0930,cinnamon"\n\n\n\n\n'

In [74]:
sorted_by_support_confidence

,antecedents,consequents,support,confidence
5,(Time:=0930),(cinnamon),0.4,1.000000
1,(Time:=0920),(Biscuit),0.3,0.500000
3,(Time:=0920),(Chocolate),0.2,0.333333


In [75]:
# sorted_by_support_confidence_col_position_changed = sorted_by_support_confidence.iloc[:, [1, 0, 2, 3]]
# sorted_by_support_confidence_col_position_changeda

In [ ]:
recommendations_to_user_OR_for_cold_starters  = ['cinnamon', 'kurudu', 'Biscuit']
finall = recommendations_to_user_OR_for_cold_starters
    #We avoided 'Chocolate'. Ie, we assume eventhough the "Time rule" is there for Chocolate, a recommendation for 'Chocolate' was not generated for the user when generating recommendations at "1 Using 'orders' to recommend items.ipynb"


perfect_time_to_show_item = dict()

for i in sorted_by_support_confidence['antecedents'].unique():
    perfect_time_to_show_item[list(i)[0]] = []


for i in sorted_by_support_confidence.values:
    time = list(i[0])[0]
    item = list(i[1])[0]
    for j in finall:
        if item == j:
            perfect_time_to_show_item[time].append(item)




In [ ]:
print("Therefore, with upto now recommendations to the user being")
print(recommendations_to_user_OR_for_cold_starters)
print("finally we show those items to user at times as")
print(perfect_time_to_show_item)

Therefore, with upto now recommendations to the user being
['cinnamon', 'kurudu', 'Biscuit']
finally we show those items to user at times as
{'Time:=0930': ['cinnamon'], 'Time:=0920': ['Biscuit']}


In [78]:
print("But still there can be having recommendations which has no Time rule noh. We add them too to the dictionary as 'Undecided' ")

But still there can be having recommendations which has no Time rule noh. We add them too to the dictionary as 'Undecided' 


In [79]:
list_who_did_not_got_times = finall


unique_items_who_got_times = []
for j in perfect_time_to_show_item.values():
    for k in j:
        unique_items_who_got_times.append(k)

unique_items_who_got_times = list(dict.fromkeys(unique_items_who_got_times))
for i in unique_items_who_got_times:
    for j in list_who_did_not_got_times:
        if(i==j):
            list_who_did_not_got_times.remove(j)



In [80]:
perfect_time_to_show_item['Undecided'] = list_who_did_not_got_times
perfect_time_to_show_item

{'Time:=0930': ['cinnamon'],
 'Time:=0920': ['Biscuit'],
 'Undecided': ['kurudu']}

In [ ]:
print("Therefore, with upto now recommendations to the user being")
print(recommendations_to_user_OR_for_cold_starters, 'There is an issue in here. That is, errorly I get the items I did not remove at previous step, only.')
print("finally we show those items to user at times as")
print(perfect_time_to_show_item)

Therefore, with upto now recommendations to the user being
['kurudu'] There is an issue in here. That is, errorly I get the items I did not remove at previous step, only.
finally we show those items to user at times as
{'Time:=0930': ['cinnamon'], 'Time:=0920': ['Biscuit'], 'Undecided': ['kurudu']}
